# 🎮 Pokémon Battle Strategy — A/B Testing com Python

> **Pergunta de pesquisa:** Qual estratégia de batalha converte mais treinadores em Campeões — a estratégia física (Fogo) ou a estratégia especial (Água)?

Este notebook documenta o experimento completo: coleta de dados reais via PokéAPI, simulação das batalhas, teste estatístico e interpretação dos resultados.

---

**Autor:** Douglas Moreira  
**Data:** 2026  
**Stack:** Python · pandas · numpy · scipy · matplotlib · seaborn

## 0. Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import requests
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

print('✅ Ambiente configurado com sucesso!')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

---

## 1. Definição do Experimento

### O que é A/B Testing?

A/B Testing é um método estatístico para comparar duas versões (A e B) de algo e determinar qual performa melhor com base em dados. É amplamente utilizado em produto, marketing e UX.

### Desenho experimental

| | Grupo A (Controle) | Grupo B (Variação) |
|---|---|---|
| **Tipo** | Fogo 🔥 | Água 💧 |
| **Estratégia** | Física | Especial |
| **Atributos** | `attack` + `speed` | `special_attack` + `hp` |

### Hipóteses

- **H₀ (nula):** Não há diferença significativa na taxa de vitória entre os grupos A e B
- **H₁ (alternativa):** Um dos grupos apresenta taxa de vitória significativamente maior
- **Nível de significância:** α = 0,05

---

## 2. Coleta de Dados — PokéAPI

In [ ]:
def get_pokemon_by_type(type_name, limit=40):
    """Coleta atributos de Pokémon por tipo via PokéAPI."""
    url = f"https://pokeapi.co/api/v2/type/{type_name}"
    r = requests.get(url).json()
    pokemons = []

    for entry in r['pokemon'][:limit]:
        name = entry['pokemon']['name']
        poke = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name}").json()
        stats_dict = {s['stat']['name']: s['base_stat'] for s in poke['stats']}
        pokemons.append({
            'name': name,
            'type': type_name,
            'attack': stats_dict.get('attack', 0),
            'special_attack': stats_dict.get('special-attack', 0),
            'speed': stats_dict.get('speed', 0),
            'hp': stats_dict.get('hp', 0),
        })

    return pd.DataFrame(pokemons)

print('⏳ Coletando dados... (pode levar alguns segundos)')
fire_df = get_pokemon_by_type('fire')
water_df = get_pokemon_by_type('water')
df_raw = pd.concat([fire_df, water_df], ignore_index=True)

df_raw.to_csv('../data/pokemon_stats.csv', index=False)
print(f'✅ {len(df_raw)} Pokémon coletados e salvos em data/pokemon_stats.csv')

### 2.1 Visualização inicial dos dados

In [ ]:
print(f'Shape: {df_raw.shape}')
print(f'\nDistribuição por tipo:')
print(df_raw['type'].value_counts())
print(f'\nValores nulos:')
print(df_raw.isnull().sum())
df_raw.head(10)

In [ ]:
print('📊 Estatísticas descritivas por tipo:')
df_raw.groupby('type')[['attack', 'special_attack', 'speed', 'hp']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribuição de Atributos por Tipo', fontsize=14, fontweight='bold')

atributos = ['attack', 'special_attack', 'speed', 'hp']
cores = {'fire': '#FF6B6B', 'water': '#4DABF7'}

for ax, attr in zip(axes.flatten(), atributos):
    for tipo, grupo in df_raw.groupby('type'):
        ax.hist(grupo[attr], alpha=0.6, label=tipo.capitalize(),
                color=cores[tipo], bins=15, edgecolor='white')
    ax.set_title(attr.replace('_', ' ').title())
    ax.set_xlabel('Base Stat')
    ax.set_ylabel('Frequência')
    ax.legend()

plt.tight_layout()
plt.savefig('../outputs/distribuicao_atributos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo em outputs/')

---

## 3. Simulação do Experimento

In [ ]:
df = df_raw.copy()

# Atribuição dos grupos
df['group'] = np.where(df['type'] == 'fire', 'A', 'B')

# Cálculo do power_score por estratégia
df['power_score'] = np.where(
    df['group'] == 'A',
    df['attack'] * 0.6 + df['speed'] * 0.4,       # Estratégia Física
    df['special_attack'] * 0.6 + df['hp'] * 0.4    # Estratégia Especial
)

# Probabilidade de vitória normalizada
df['win_prob'] = df['power_score'] / df['power_score'].max()

# Simulação binomial (seed=42 garante reprodutibilidade)
df['converted'] = np.random.binomial(1, df['win_prob'])

df.to_csv('../data/simulated_battles.csv', index=False)
print('✅ Simulação concluída!')
print(f'\nTaxa de vitória bruta por grupo:')
print(df.groupby('group')['converted'].mean().round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribuição do Power Score por Grupo', fontsize=13, fontweight='bold')

cores_grupo = {'A': '#FF6B6B', 'B': '#4DABF7'}
labels = {'A': 'Grupo A — Físico (Fogo)', 'B': 'Grupo B — Especial (Água)'}

for ax, (grupo, dados) in zip(axes, df.groupby('group')):
    ax.hist(dados['power_score'], bins=15, color=cores_grupo[grupo],
            alpha=0.8, edgecolor='white')
    ax.axvline(dados['power_score'].mean(), color='black',
               linestyle='--', label=f'Média: {dados["power_score"].mean():.1f}')
    ax.set_title(labels[grupo])
    ax.set_xlabel('Power Score')
    ax.set_ylabel('Frequência')
    ax.legend()

plt.tight_layout()
plt.savefig('../outputs/power_score_distribuicao.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 4. Teste Estatístico — t-test de Welch

Utilizamos o **t-test de Welch** (`ttest_ind` com `equal_var=False`), adequado para comparar duas amostras independentes sem assumir variâncias iguais.

In [ ]:
group_a = df[df['group'] == 'A']['converted']
group_b = df[df['group'] == 'B']['converted']

rate_a = group_a.mean()
rate_b = group_b.mean()
lift = (rate_b - rate_a) / rate_a * 100 if rate_a > 0 else 0

t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)

print('=' * 50)
print('        RESULTADO DO A/B TEST')
print('=' * 50)
print(f'  Grupo A (Físico/Fogo):    {rate_a:.2%}')
print(f'  Grupo B (Especial/Água):  {rate_b:.2%}')
print(f'  Lift (B vs A):            {lift:+.1f}%')
print(f'  T-statistic:              {t_stat:.4f}')
print(f'  p-value:                  {p_value:.4f}')
print('=' * 50)

alpha = 0.05
if p_value < alpha:
    winner = 'A (Físico/Fogo)' if rate_a > rate_b else 'B (Especial/Água)'
    print(f'\n✅ Diferença SIGNIFICATIVA (p < {alpha})')
    print(f'   Vencedor: Grupo {winner}')
else:
    print(f'\n⚠️  Sem evidência estatística suficiente (p ≥ {alpha})')
    print(f'   Falha em rejeitar H₀')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Resultado do A/B Test — Estratégias de Batalha Pokémon', fontsize=13, fontweight='bold')

# Gráfico 1: Taxa de vitória
grupos = ['A\n(Físico · Fogo)', 'B\n(Especial · Água)']
taxas = [rate_a, rate_b]
cores_bar = ['#FF6B6B', '#4DABF7']

bars = axes[0].bar(grupos, taxas, color=cores_bar, alpha=0.85,
                   edgecolor='white', linewidth=1.5, width=0.5)
for bar, taxa in zip(bars, taxas):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{taxa:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_title('Taxa de Vitória por Grupo')
axes[0].set_ylabel('Taxa de Vitória')
axes[0].set_ylim(0, 1)
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='50%')
axes[0].legend()

# Gráfico 2: p-value
axes[1].barh(['p-value'], [p_value], color='#74C69D' if p_value < 0.05 else '#FF6B6B', alpha=0.85)
axes[1].axvline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
axes[1].set_title('p-value vs Nível de Significância')
axes[1].set_xlabel('p-value')
axes[1].set_xlim(0, max(0.1, p_value * 1.5))
axes[1].text(p_value + 0.001, 0, f'{p_value:.4f}', va='center', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/ab_test_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico final salvo em outputs/ab_test_result.png')

---

## 5. Conclusão

### Interpretação dos Resultados

| Critério | Valor | Interpretação |
|---|---|---|
| p-value | — | Preencher após execução |
| Decisão | — | Rejeitar H₀ ou Falha em rejeitar |
| Grupo vencedor | — | A, B ou inconclusivo |

### Limitações do experimento

- A simulação usa probabilidade baseada em atributos base, sem considerar movimentos, habilidades ou vantagens de tipo
- A amostra é limitada a 40 Pokémon por tipo — aumentar o `limit` na coleta pode alterar os resultados
- O modelo de `power_score` é simplificado; pesos (0.6 / 0.4) foram definidos arbitrariamente

### Próximos passos

- Testar outros pares de tipos (ex.: Elétrico vs. Grama)
- Adicionar cálculo do **intervalo de confiança** para a diferença entre taxas
- Incorporar **tamanho de efeito** (Cohen's d) para além do p-value
- Aplicar a mesma metodologia a dados reais de campanhas ou experimentos de produto

---

*Projeto desenvolvido como parte de um portfólio de Data Science com foco em análise estatística aplicada.*